## Apps in prod with payment task

In [ ]:
with Apps.init() as apps:
    apps.where(lambda app: app.env == "prod").where(
        lambda app: app.process.tasks.some(lambda task: task.type == "payment")
    ).select({"Frontend version": lambda app: app.frontend_version, "Repo": lambda app: app.repo_url}).csv().table()

## Different process flows in prod

In [ ]:
with Apps.init() as apps:
    # apps.where(lambda app: app.process.tasks.some(lambda task: task.type == "signing")).table()
    apps.where(lambda app: app.env == "prod").group_by(
        {
            "Process": lambda app: app.process.tasks.map(lambda task: task.type or "Unknown task type")
            # .filter(lambda task_type: task_type != None)
            .reduce(lambda a, b: f"{a}->{b}")
            or "No process"
        }
    ).select({"Count": lambda apps: apps.length}).order_by(lambda apps: apps.length, True).table()

In [ ]:
with Apps.init() as apps:
    # apps.where(lambda app: app.process.tasks.some(lambda task: task.type == "signing")).table()
    apps.where(lambda app: app.env == "prod").unique_repos().select(
        {
            "Process": lambda app: app.process.tasks.map(lambda task: task.type or "Unknown task type")
            # .filter(lambda task_type: task_type != None)
            .reduce(lambda a, b: f"{a}->{b}") or "No process",
            "Repo": lambda app: app.repo_url,
        }
    ).order_by(lambda app: (app["Process"], app.org, app.app)).csv("process").table()